# 参数管理

In [2]:
import torch
from torch import nn

net=nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1))
X=torch.rand(size=(2,4))
net(X)


tensor([[-0.0026],
        [ 0.0142]], grad_fn=<AddmmBackward0>)

squential 实际上可以理解为一个python列表，上面我们定义了一个简单的MLP模型结构nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1)；这里对应的索引就是0,1,2

In [3]:
print(net[2].state_dict())# net[2]实际上就是最后的隐藏层

OrderedDict([('weight', tensor([[-0.3191,  0.2670, -0.0620, -0.1849, -0.2412,  0.1530,  0.2837, -0.2424]])), ('bias', tensor([-0.0069]))])


## 目标参数
我们可以直接将对应位置处的参数值给取出来

In [ ]:
type(net[2].bias),net[2].bias,net[2].bias.data
"""从输出可以看到参数的类型是pytorch中特殊的Parameter类型，我们直接打印的时候他会显示各项内容，我们可以使用.data来彻底的访问数据"""

(torch.nn.parameter.Parameter,
 Parameter containing:
 tensor([-0.0069], requires_grad=True),
 tensor([-0.0069]))

In [6]:
net[2].weight.grad==None,net[2].bias.grad==None

(True, True)

一次性访问全部元素，可以使用 named_parameters() 

In [10]:
print(*[(name,param.shape) for name,param in net[2].named_parameters()])#只取出来最后一个线性层的参数，前面的 * python中是解包，将列表拆开
print(*[(name,param.shape) for name,param in net.named_parameters()])#这里是将整个网络的参数和名称都取出来
print([(name,param.shape) for name,param in net[2].named_parameters()])#不解包就是一个列表

('weight', torch.Size([1, 8])) ('bias', torch.Size([1]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))
[('weight', torch.Size([1, 8])), ('bias', torch.Size([1]))]


In [11]:
net.state_dict()["2.bias"].data

tensor([-0.0069])

从嵌套块中获得参数

In [ ]:
def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,4),nn.ReLU())

def block2():
    """使用块来构建模型层内参数

    :return: 返回一个net
    """
    net=nn.Sequential()

    for i in range(5):
        net.add_module(f'block{i}',block1())
    return net

regnet=nn.Sequential(block2(),nn.Linear(4,1))
regnet(X)

tensor([[-0.4279],
        [-0.4279]], grad_fn=<AddmmBackward0>)

In [15]:
print(regnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block4): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features

内置初始化

In [ ]:
def init_normal(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
net[0].weight.data[0],net[0].bias.data[0]

(tensor([-0.0032,  0.0108,  0.0038,  0.0139]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0.]))

In [19]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0],net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

可以对不同的块使用不同的初始化方法

In [21]:
def xavier(m):
    if type(m)==nn.Linear:
        nn.init.xavier_normal_(m.weight)

def init_42(m):
    if type(m)==nn.Linear:
        nn.init.constant_(m.weight,42)

net[0].apply(xavier)
net[2].apply(init_42)
net[0].weight.data[0],net[2].weight.data[0]

(tensor([-0.4383, -0.0528,  0.2647, -0.2207]),
 tensor([42., 42., 42., 42., 42., 42., 42., 42.]))

自定义初始化

In [28]:
def my_init(m):
    if type(m)==nn.Linear:
        print(*[(name,param.shape) for name,param in m.named_parameters()][0])
        nn.init.uniform_(m.weight,-10,10)
        m.weight.data*=m.weight.data.abs()>=5
net.apply(my_init)
net[0].weight[:2]

weight torch.Size([8, 4])
weight torch.Size([1, 8])


tensor([[ 7.4298,  9.9123, -0.0000, -6.0127],
        [-0.0000,  0.0000, -0.0000,  7.9007]], grad_fn=<SliceBackward0>)

In [ ]:
net[0].weight.data[:]+=1
net[0].weight.data 

tensor([[ 8.4298, 10.9123,  1.0000, -5.0127],
        [ 1.0000,  1.0000,  1.0000,  8.9007],
        [-8.7302,  1.0000,  1.0000,  1.0000],
        [ 8.9407,  1.0000,  1.0000,  1.0000],
        [ 1.0000, -8.9852,  7.3500,  8.5295],
        [ 1.0000,  1.0000,  1.0000,  1.0000],
        [ 1.0000, -7.1407,  6.4933, -7.4025],
        [-7.6756, -7.4508,  1.0000,  1.0000]])

参数绑定

In [34]:
shared=nn.Linear(8,8)

net=nn.Sequential(nn.Linear(4,8),nn.ReLU(),shared,nn.ReLU(),shared,nn.ReLU(),nn.Linear(8,1))

print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0,0]=100
net[2].weight.data[0] == net[4].weight.data[0]

tensor([True, True, True, True, True, True, True, True])


tensor([True, True, True, True, True, True, True, True])